In [5]:
import numpy as np
import pandas as pd
import os
import gc
from collections import defaultdict

# KHỞI TẠI THƯ VIỆN NGÀY LỄ CỦA MỸ (Để tạo biếnis_holiday)
# Nếu máy bạn chưa có, gõ lệnh dưới Terminal: pip install holidays
import holidays
us_holidays = holidays.US(years=2021)

def calculate_kpi_metrics(df):
    if len(df) == 0:
        return 0, 0, 0
    trips = len(df)
    revenue = df["total_amount"].sum()
    speed_median = df["speed_mph"].median() if "speed_mph" in df.columns else 0
    return trips, revenue, speed_median

def clean_month(month, soft_percentiles=True):
    # CẬP NHẬT: Sửa đường dẫn lùi 3 tầng cho đúng thực tế thư mục máy bạn
    file = f"../../../Data/raw/yellow_tripdata_2021-{month:02d}.parquet"
    print(f"\n--- Đang xử lý: {file} ---")
    df = pd.read_parquet(file)

    # -----------------------------
    # 1. Chuẩn hoá datetime & Trích xuất Biến Thời Gian Động (ĐÃ THÊM FLAG NGÀY LỄ)
    # -----------------------------
    df["tpep_pickup_datetime"]  = pd.to_datetime(df["tpep_pickup_datetime"], errors="coerce")
    df["tpep_dropoff_datetime"] = pd.to_datetime(df["tpep_dropoff_datetime"], errors="coerce")
    
    # Ép kiểu tối ưu bảo vệ RAM
    df["pickup_hour"] = df["tpep_pickup_datetime"].dt.hour.astype("uint8")
    df["day_of_week"] = df["tpep_pickup_datetime"].dt.dayofweek.astype("uint8")
    
    # THÊM MỚI: Biến Flag Ngày lễ (1: Ngày lễ, 0: Ngày thường)
    pickup_dates = df["tpep_pickup_datetime"].dt.date
    df["is_holiday"] = pickup_dates.isin(us_holidays).astype("uint8")

    # -----------------------------
    # 2. Chuẩn hoá numeric
    # -----------------------------
    numeric_cols = [
        "passenger_count", "trip_distance", "fare_amount", "extra", "mta_tax",
        "tip_amount", "tolls_amount", "improvement_surcharge", "total_amount",
        "congestion_surcharge", "airport_fee", "RatecodeID"
    ]
    for c in numeric_cols:
        df[c] = pd.to_numeric(df[c], errors="coerce")

    # -----------------------------
    # 3. Trip duration & Tốc độ
    # -----------------------------
    df["trip_duration"] = (df["dropoff_datetime" if "dropoff_datetime" in df.columns else "tpep_dropoff_datetime"] - df["tpep_pickup_datetime"]).dt.total_seconds() / 60
    df["speed_mph"] = df["trip_distance"] / (df["trip_duration"] / 60)

    # -----------------------------
    # 4. Tách Flex Fare & Giải phóng RAM
    # -----------------------------
    df["is_flex_fare"] = (df["payment_type"] == 0)
    metered = df[~df["is_flex_fare"]].copy()
    flex    = df[df["is_flex_fare"]].copy()
    del df
    gc.collect()

    # -----------------------------
    # 5. Hard QA & Soft QA (Giữ nguyên logic chuẩn của bạn)
    # -----------------------------
    # Đọc trực tiếp từ link trực tuyến để tránh lỗi lệch đường dẫn giữa các máy thành viên
    lookup = pd.read_csv("https://d37ci6vzurychx.cloudfront.net/misc/taxi_zone_lookup.csv")
    valid_ids = lookup["LocationID"].unique()

    metered["valid_vendor"] = metered["VendorID"].isin([1,2,6,7])
    metered["valid_time"]   = (metered["tpep_dropoff_datetime"] >= metered["tpep_pickup_datetime"]) & \
                              (metered["trip_duration"] > 0) & (metered["tpep_pickup_datetime"].dt.year == 2021)
    metered["valid_passenger"] = metered["passenger_count"].isin([1,2,3,4,5,6,99])
    metered["valid_ratecode"]  = metered["RatecodeID"].isin([1,2,3,4,5,6])
    metered["valid_store"]     = metered["store_and_fwd_flag"].isin(["Y","N"])
    metered["valid_payment"]   = metered["payment_type"].isin([1,2,3,4,5,6])
    metered["valid_PULocationID"] = metered["PULocationID"].isin(valid_ids)
    metered["valid_DOLocationID"] = metered["DOLocationID"].isin(valid_ids)
    metered["valid_speed"] = (metered["speed_mph"] >= 1) & (metered["speed_mph"] <= metered["speed_mph"].quantile(0.999))

    if soft_percentiles:
        for col in ["trip_distance","trip_duration","fare_amount"]:
            lower = metered[col].quantile(0.0001)
            upper = metered[col].quantile(0.9999)
            metered[f"valid_{col}"] = (metered[col] >= lower) & (metered[col] <= upper)
        price_per_mile = metered["fare_amount"] / metered["trip_distance"].replace(0, 0.001)
        metered["valid_economics"] = ~((metered["trip_distance"] > 5) & (price_per_mile < 0.5))

    metered["valid_tip"] = ((metered["payment_type"] == 1) & (metered["tip_amount"] >= 0)) | \
                            ((metered["payment_type"] != 1) & (metered["tip_amount"] == 0))
    metered["valid_extra"] = metered["extra"] >= 0
    metered["valid_mta"]   = metered["mta_tax"].isin([0,0.5])
    metered["valid_improvement"] = metered["improvement_surcharge"].isin([0,0.3])
    metered["valid_congestion"]  = metered["congestion_surcharge"].isin([0,0.75,2.5])
    metered["valid_tolls"] = metered["tolls_amount"] >= 0
    metered["valid_airport"] = metered["airport_fee"].fillna(0) >= 0

    flag_cols = [c for c in metered.columns if c.startswith("valid_")]
    metered["valid_all"] = metered[flag_cols].all(axis=1)
    
    total_rows = len(metered)
    qa_summary = []
    for col in flag_cols:
        fail_cnt = (~metered[col]).sum()
        qa_summary.append({"rule": col, "fail_count": fail_cnt})
    qa_summary_df = pd.DataFrame(qa_summary)

    clean = metered[metered["valid_all"]].copy()
    bad   = metered[~metered["valid_all"]].copy()

    # PHÂN TÍCH TÁC ĐỘNG KPI
    raw_trips, raw_rev, raw_speed = calculate_kpi_metrics(metered)
    clean_trips, clean_rev, clean_speed = calculate_kpi_metrics(clean)

    impact_data = {
        "month": month,
        "raw_revenue": raw_rev,
        "clean_revenue": clean_rev,
        "revenue_risk": raw_rev - clean_rev,
        "raw_speed_p50": raw_speed,
        "clean_speed_p50": clean_speed,
        "speed_diff": raw_speed - clean_speed
    }

    # -----------------------------
    # 6. THÊM MỚI: TẠO MA TRẬN ĐẶC TRƯNG KHÔNG - THỜI GIAN SIÊU MỊN (AGGREGATION)
    # -----------------------------
    # Nhóm dữ liệu theo: Zone + Giờ + Thứ + Ngày lễ để tính đặc trưng
    df_time_features_month = clean.groupby(
        ["PULocationID", "pickup_hour", "day_of_week", "is_holiday"]
    ).agg(
        total_trips=("total_amount", "count"),          # Tổng số chuyến
        avg_revenue=("total_amount", "mean"),           # Doanh thu trung bình/chuyến
        median_speed=("speed_mph", "median"),           # Tốc độ trung vị luồng xe
        avg_duration=("trip_duration", "mean"),         # Thời gian di chuyển trung bình
        avg_distance=("trip_distance", "mean")          # Quãng đường trung bình
    ).reset_index()

    # Downcast (Ép kiểu dữ liệu nhỏ nhất có thể) để tối ưu RAM tối đa
    df_time_features_month["total_trips"] = pd.to_numeric(df_time_features_month["total_trips"], downcast="unsigned")
    for col in ["avg_revenue", "median_speed", "avg_duration", "avg_distance"]:
        df_time_features_month[col] = pd.to_numeric(df_time_features_month[col], downcast="float")

    # -----------------------------
    # 7. Lưu trữ dữ liệu hệ thống (Đường dẫn lùi 3 tầng thực tế)
    # -----------------------------
    os.makedirs("../../Data/processed/clean_data", exist_ok=True)
    os.makedirs("../../Data/processed/bad_data", exist_ok=True)
    os.makedirs("../../Data/processed/flex_fare", exist_ok=True)
    os.makedirs("../../Data/processed/time_features", exist_ok=True)

    clean_path = f"../../Data/processed/clean_data/clean_2021-{month:02d}.parquet"
    bad_path   = f"../../Data/processed/bad_data/bad_2021-{month:02d}.parquet"
    flex_path  = f"../../Data/processed/flex_fare/flex_2021-{month:02d}.parquet"
    feature_path = f"../../Data/processed/time_features/features_2021-{month:02d}.parquet"

    # Chỉ lưu các cột lõi ở tệp giao dịch sạch
    cols_to_save = ["PULocationID", "pickup_hour", "day_of_week", "is_holiday", "trip_duration", "trip_distance", "total_amount", "speed_mph"]
    clean[cols_to_save].to_parquet(clean_path, index=False)
    bad.to_parquet(bad_path, index=False)
    flex.to_parquet(flex_path, index=False)
    
    # Lưu file ma trận đặc trưng siêu mịn của tháng
    df_time_features_month.to_parquet(feature_path, index=False)

    del metered, clean, bad, flex, df_time_features_month
    gc.collect()

    return len(qa_summary_df), qa_summary_df, total_rows, impact_data

# ==========================================================
# VÒNG LẶP VẬN HÀNH TOÀN CỤC 12 THÁNG
# ==========================================================
total_rows_all = 0
rule_fail_all = defaultdict(int)
impact_results = []

for month in range(1, 13):
    _, qa_df, total_rows, impact_data = clean_month(month)
    impact_results.append(impact_data)
    total_rows_all += total_rows

    for _, row in qa_df.iterrows():
        rule_fail_all[row["rule"]] += row["fail_count"]

# -----------------------------
# 8. THÊM MỚI: GỘP TOÀN BỘ 12 THÁNG THÀNH BẢNG TỔNG df_time_features KHÔNG LÀM TRÀN RAM
# -----------------------------
print("\n--- Đang tổng hợp Ma trận Đặc trưng Không - Thời gian 12 tháng... ---")
import glob
feature_files = glob.glob("../../Data/processed/time_features/features_2021-*.parquet")

# Gộp các bảng tổng hợp nhỏ (đã rất nhẹ) thành bảng tổng siêu mịn của cả năm
df_time_features = pd.concat([pd.read_parquet(f) for f in feature_files], ignore_index=True)

# Gom nhóm lại một lần cuối cùng trên quy mô năm
df_time_features = df_time_features.groupby(
    ["PULocationID", "pickup_hour", "day_of_week", "is_holiday"]
).agg(
    total_trips=("total_trips", "sum"),
    avg_revenue=("avg_revenue", "mean"),
    median_speed=("median_speed", "mean"),
    avg_duration=("avg_duration", "mean"),
    avg_distance=("avg_distance", "mean")
).reset_index()

# Xuất sản phẩm cuối cùng bàn giao cho Thành viên 2 và Thành viên 4
output_final_path = "../../Data/processed/df_time_features.parquet"
df_time_features.to_parquet(output_final_path, index=False)
print(f"✅ ĐÃ XUẤT THÀNH CÔNG SẢN PHẨM CUỐI CÙNG: {output_final_path}")

# In báo cáo chất lượng cho văn bản báo cáo của bạn
qa_year = pd.DataFrame([
    {"rule": rule, "fail_count": cnt, "fail_rate_percent": cnt / total_rows_all * 100}
    for rule, cnt in rule_fail_all.items()
]).sort_values("fail_rate_percent", ascending=False)

print("\n=== QA SUMMARY - FULL YEAR (2021) ===")
print(qa_year)

impact_df = pd.DataFrame(impact_results)
impact_df.to_csv("qa_impact_analysis.csv", index=False)
print("\n=== ĐÃ XUẤT FILE qa_impact_analysis.csv THÀNH CÔNG ===")


--- Đang xử lý: ../../../Data/raw/yellow_tripdata_2021-01.parquet ---

--- Đang xử lý: ../../../Data/raw/yellow_tripdata_2021-02.parquet ---

--- Đang xử lý: ../../../Data/raw/yellow_tripdata_2021-03.parquet ---

--- Đang xử lý: ../../../Data/raw/yellow_tripdata_2021-04.parquet ---

--- Đang xử lý: ../../../Data/raw/yellow_tripdata_2021-05.parquet ---

--- Đang xử lý: ../../../Data/raw/yellow_tripdata_2021-06.parquet ---

--- Đang xử lý: ../../../Data/raw/yellow_tripdata_2021-07.parquet ---

--- Đang xử lý: ../../../Data/raw/yellow_tripdata_2021-08.parquet ---

--- Đang xử lý: ../../../Data/raw/yellow_tripdata_2021-09.parquet ---

--- Đang xử lý: ../../../Data/raw/yellow_tripdata_2021-10.parquet ---

--- Đang xử lý: ../../../Data/raw/yellow_tripdata_2021-11.parquet ---

--- Đang xử lý: ../../../Data/raw/yellow_tripdata_2021-12.parquet ---

--- Đang tổng hợp Ma trận Đặc trưng Không - Thời gian 12 tháng... ---
✅ ĐÃ XUẤT THÀNH CÔNG SẢN PHẨM CUỐI CÙNG: ../../Data/processed/df_time_feature

In [6]:
import os
print("Danh sách file trong time_features:")
try:
    print(os.listdir("../../Data/processed/time_features"))
except Exception as e:
    print("Chưa tìm thấy thư mục hoặc lỗi:", e)

Danh sách file trong time_features:
['features_2021-01.parquet', 'features_2021-02.parquet', 'features_2021-03.parquet', 'features_2021-04.parquet', 'features_2021-05.parquet', 'features_2021-06.parquet', 'features_2021-07.parquet', 'features_2021-08.parquet', 'features_2021-09.parquet', 'features_2021-10.parquet', 'features_2021-11.parquet', 'features_2021-12.parquet']


In [7]:
import pandas as pd
df_test = pd.read_parquet("../../Data/processed/df_time_features.parquet")
print("Kích thước ma trận không-thời gian:", df_test.shape)
display(df_test.head())

Kích thước ma trận không-thời gian: (45300, 9)


,PULocationID,pickup_hour,day_of_week,is_holiday,total_trips,avg_revenue,median_speed,avg_duration,avg_distance
0,1,0,4,0,1,87.660004,25.563625,54.616665,23.270000
1,1,0,5,0,1,130.199997,26.571127,47.216667,20.910000
2,1,1,3,0,4,114.248329,30.925545,37.011112,21.258333
3,1,1,3,1,1,84.660004,38.846153,31.200001,20.200001
4,1,4,0,0,1,90.800003,9.818182,0.183333,0.030000
